# Cross-validation

When we split the data into a training set and a test set just once, the performance we measure depends on the particular sampling we made. If we stop fixing the sampling seed (`random_state` parameter of the `train_test_split` function), we will get different results in each run. And if we use the test set to compare models, we fall into [model selection leakage](data_leakage.ipynb#Model-selection-leakage).

A more efficient approach is **cross-validation**: instead of splitting the training set in two, it is divided into *k* sets (*folds*). Then the model is trained *k* times, each time leaving a different fold as the validation set and the other *k-1* as the training set. The result is an array with *k* scores. The results are similar to what we could obtain by running the code *k* times without fixing the random sampling seed. The trade-off is obvious: the computational cost is also multiplied by *k*.

This introduces the concept of the **validation set**: the data used to compare models and **tune hyperparameters**, which changes in each iteration of cross-validation. The test set is reserved for evaluating the final performance of the chosen model once it has been trained.

[<img src="../img/cross-validation.png" width="500">](https://www.researchgate.net/figure/Train-test-cross-validation-split-methodology-used-in-this-paper-The-first-operation_fig2_340567535)

[<img src="../img/cross-validation2.png" width="700">](https://www.statology.org/validation-set-vs-test-set/)

## Loading the data and a model to evaluate

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

housing = pd.read_csv("../data/housing.csv")
train_set, test_set = train_test_split(
    housing, test_size=0.2, random_state=42,
    stratify=pd.cut(housing["median_income"], bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5]))

X_train = train_set.drop("median_house_value", axis=1)  # features
y_train = train_set["median_house_value"].copy()        # target (labels)

In [ ]:
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

preprocessing = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), make_column_selector(dtype_include=np.number)),
    (OneHotEncoder(handle_unknown="ignore"), make_column_selector(dtype_include=object)),
)
tree_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))

## `cross_val_score`

The following code trains and evaluates the model with 10 different samplings of the training set.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

# shuffle=True guarantees random folds, preventing bias from ordered data.
# random_state=42 guarantees the same folds every run, ensuring fair model comparison.
cv = KFold(n_splits=10, shuffle=True, random_state=42)

tree_rmses = -cross_val_score(estimator=tree_reg,
                              X=X_train,
                              y=y_train,
                              scoring="neg_root_mean_squared_error",
                              cv=cv)
print(tree_rmses)
pd.Series(tree_rmses).describe()

[67916.96186611 69455.5750236  71354.20226516 68675.13014606
 70120.78943944 68703.42874049 69577.19063471 73031.36096637
 65956.84555273 70645.6617551 ]


count       10.000000
mean     69543.714639
std       1942.798903
min      65956.845553
25%      68682.204795
50%      69516.382829
75%      70514.443676
max      73031.360966
dtype: float64

The `scoring` parameter of the `cross_val_score` function expects a **utility function** (higher is better) rather than a **cost function** (lower is better), so the score is actually the negative of the RMSE. `cross_val_score` will seek to maximize the score, so by maximizing the negative of the RMSE, we minimize the RMSE; we flip the sign of the output to get the RMSE values.

There are multiple string identifiers for evaluation metrics that can be used in `scoring`, which can be found in the [Scikit-Learn documentation](https://scikit-learn.org/stable/modules/model_evaluation.html#scoring-parameter).

The mean and standard deviation of the 10 RMSEs provide more detailed information (and less dependent on the sampling we performed) about the model's performance than a single number. Now we have an evaluation metric for our decision tree model that we can compare with others without touching the test set.

## Shuffled folds

By default, passing an integer (e.g., `cv=10`) to `cross_val_score` for a regression task uses `KFold` *without* shuffling. This separates the dataset into sequential chunks. If the data has any inherent order (e.g., sorted by price or date), this can lead to biased folds where the training set looks very different from the validation set.

To ensure robust and unbiased evaluation, we should explicitly use a **shuffled K-Fold**. This guarantees that each fold is a random representative sample of the whole dataset. Additionally, setting a fixed `random_state` ensures reproducibility, so we can fairly compare different models on the exact same data splits. For classification, `StratifiedKFold` additionally keeps the class proportions in every fold (see [Stratified sampling](stratified_sampling.ipynb)).

## Comparing another model

*Random Forest* is a model that trains multiple decision trees (***ensemble learning***) on random subsets of the data and features and averages their predictions. Evaluating it with the *same* `cv` object means both models are scored on exactly the same folds.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = make_pipeline(preprocessing, RandomForestRegressor(random_state=42))
forest_rmses = -cross_val_score(forest_reg, X_train, y_train, scoring="neg_root_mean_squared_error", cv=cv)
pd.Series(forest_rmses).describe()

count       10.000000
mean     49522.282161
std       2063.102407
min      46087.985020
25%      48037.671864
50%      49931.688214
75%      50717.431360
max      53230.032873
dtype: float64

The random forest achieves a clearly lower mean validation RMSE than the single decision tree. Assuming we stick with it, we could finally train the chosen model on the entire training set and evaluate its performance, once, on the test set we kept separate. Before that, its hyperparameters can be tuned with the same cross-validation machinery: see [Hyperparameter search](hyperparameter_search.ipynb).